In [1]:
import pandas as pd

### payment collection rate for active clients

In [3]:
#read in files
client_df = pd.read_csv('client_data_20250423120102.csv')
repayment_df = pd.read_csv('repayment_data_20250423120137.csv')
reconciled_df = pd.read_csv('reconciled_payment_data_20250423120155.csv')

#standardize column headers
client_df.columns = client_df.columns.str.lower().str.replace(' ', '_')
reconciled_df.columns = reconciled_df.columns.str.lower().str.replace(' ', '_')
repayment_df.columns = repayment_df.columns.str.lower().str.replace(' ', '_')
print(reconciled_df.columns)
print(repayment_df.columns)
print(client_df.columns)

Index(['id', 'date', 'amount', 'type', 'note', 'payment_wallet_name',
       'destination_type', 'destination_contract_reference',
       'origin_payment_id', 'origin_payment_transaction_id',
       'payment_wallet_id', 'lead_id', 'add_on_id', 'contract_payment_id',
       'contract_pending_payment_id', 'user_id', 'client_id', 'last_updated'],
      dtype='object')
Index(['id', 'contract_id', 'contract_reference', 'client_id', 'payment_date',
       'expected_payment_date', 'days_late', 'total_amount', 'amount_paid',
       'amount_of_discount', 'credit_value', 'type', 'last_updated'],
      dtype='object')
Index(['id', 'custom_id', 'first_name', 'family_name', 'birthdate', 'age',
       'gender', 'picture_uuid', 'language_sms', 'language_spoken',
       'longitude', 'latitude', 'primary_phone_number', 'all_phone_numbers',
       'start_date', 'end_date', 'has_active_contracts',
       'has_completed_contracts', 'has_defaulted_contracts',
       'has_late_contracts', 'client_group_id',

In [4]:
#method2

In [5]:
#link repayment_data to client_data to get guardian (user_in_charge_id)
repayment_with_user_id = pd.merge(
    repayment_df[['contract_reference', 'client_id', 'amount_paid']],
    client_df[['id', 'user_in_charge_id']],
    left_on='client_id',
    right_on='id',
    how='left'
)

In [6]:
#sum expected payments per contract (now with user ID (guardian))
#using amount paid instead of total amount as there are some clients that had discount applied n didnt have to pay
expected_payments = repayment_with_user_id.groupby(
    ['contract_reference', 'user_in_charge_id']
)['amount_paid'].sum().reset_index()
expected_payments

,contract_reference,user_in_charge_id,amount_paid
0,C1000017,1.0,490000.0
1,C1010016,1.0,455000.0
2,C1020015,1.0,630000.0
3,C1030014,1.0,595000.0
4,C1040013,1.0,595000.0
...,...,...,...
931,C970012,1.0,145000.0
932,C9710013,7.0,105000.0
933,C9740010,7.0,35000.0
934,C980011,1.0,490000.0


In [7]:
#sum reconciled payments per client
reconciled_df['amount'] = reconciled_df['amount'] * -1
reconciled_payments = reconciled_df.groupby(
    'destination_contract_reference'
)['amount'].sum().reset_index()
reconciled_payments

,destination_contract_reference,amount
0,126,175000.0
1,440,175000.0
2,66,95000.0
3,C1000017,490000.0
4,C1010016,455000.0
...,...,...
934,C970012,145000.0
935,C9710013,105000.0
936,C9740010,35000.0
937,C980011,490000.0


In [8]:
#change column header so merge will work
reconciled_payments.rename(columns={'destination_contract_reference': 'contract_reference'}, inplace=True)

In [9]:
#merge expected and reconciled payments (now with user id)
merged_df = pd.merge(
    expected_payments,
    reconciled_payments,
    on='contract_reference',
    how='left'
).fillna(0)
merged_df.head(10)

,contract_reference,user_in_charge_id,amount_paid,amount
0,C1000017,1.0,490000.0,490000.0
1,C1010016,1.0,455000.0,455000.0
2,C1020015,1.0,630000.0,630000.0
3,C1030014,1.0,595000.0,595000.0
4,C1040013,1.0,595000.0,595000.0
5,C1050012,1.0,490000.0,490000.0
6,C1060011,1.0,420000.0,420000.0
7,C1070010,1.0,455000.0,455000.0
8,C1080019,1.0,760000.0,760000.0
9,C1090018,1.0,490000.0,490000.0


In [10]:
#rename for better readability
merged_df.rename(columns={
    'amount_paid': 'expected_amount',
    'amount': 'amount_collected'
}, inplace=True)
merged_df

,contract_reference,user_in_charge_id,expected_amount,amount_collected
0,C1000017,1.0,490000.0,490000.0
1,C1010016,1.0,455000.0,455000.0
2,C1020015,1.0,630000.0,630000.0
3,C1030014,1.0,595000.0,595000.0
4,C1040013,1.0,595000.0,595000.0
...,...,...,...,...
931,C970012,1.0,145000.0,145000.0
932,C9710013,7.0,105000.0,105000.0
933,C9740010,7.0,35000.0,35000.0
934,C980011,1.0,490000.0,490000.0


In [11]:
merged_df['user_in_charge_id'].unique()

array([ 1.,  7., 10.,  9., 16., 14.,  5.])

In [12]:
merged_df['user_in_charge_id'].value_counts()

user_in_charge_id
7.0     432
1.0     265
9.0     121
16.0     60
14.0     35
5.0      12
10.0     11
Name: count, dtype: int64

In [13]:
#calculate collection rate by user ID
guardian_performance = merged_df.groupby('user_in_charge_id').agg(
    actual_payments=('amount_collected', 'sum'),
    expected_payments=('expected_amount', 'sum')
).reset_index()

guardian_performance['collection_rate'] = (guardian_performance['actual_payments'] / guardian_performance['expected_payments']) * 100
guardian_performance

,user_in_charge_id,actual_payments,expected_payments,collection_rate
0,1.0,112058994.0,112018994.0,100.035708
1,5.0,5115000.0,5110000.0,100.097847
2,7.0,155600210.0,155600210.0,100.000000
3,9.0,50365044.0,50350044.0,100.029791
4,10.0,3161000.0,3161000.0,100.000000
5,14.0,17181035.0,17181035.0,100.000000
6,16.0,24836050.0,24836050.0,100.000000


In [14]:
#handle division by zero (unpaid contracts)
guardian_performance['collection_rate'] = guardian_performance.apply(
    lambda x: x['collection_rate'] if x['expected_payments'] > 0 else 0,
    axis=1)
guardian_performance

,user_in_charge_id,actual_payments,expected_payments,collection_rate
0,1.0,112058994.0,112018994.0,100.035708
1,5.0,5115000.0,5110000.0,100.097847
2,7.0,155600210.0,155600210.0,100.000000
3,9.0,50365044.0,50350044.0,100.029791
4,10.0,3161000.0,3161000.0,100.000000
5,14.0,17181035.0,17181035.0,100.000000
6,16.0,24836050.0,24836050.0,100.000000


In [15]:
#rank user in charge
guardian_performance['performance_rank'] = guardian_performance['collection_rate'].rank(ascending=False)
guardian_performance.sort_values('collection_rate', ascending=False, inplace=True)
guardian_performance

,user_in_charge_id,actual_payments,expected_payments,collection_rate,performance_rank
1,5.0,5115000.0,5110000.0,100.097847,1.0
0,1.0,112058994.0,112018994.0,100.035708,2.0
3,9.0,50365044.0,50350044.0,100.029791,3.0
2,7.0,155600210.0,155600210.0,100.000000,5.5
4,10.0,3161000.0,3161000.0,100.000000,5.5
5,14.0,17181035.0,17181035.0,100.000000,5.5
6,16.0,24836050.0,24836050.0,100.000000,5.5


In [16]:
#method 3
repayment_df['type'].unique()

array([nan, 'Reversed Contract Payment', 'Manual Discount',
       'Manual Delay', 'Reversed Downpayment', 'Downpayment',
       'Offer Change Delay'], dtype=object)

In [17]:
repayment_df = repayment_df[repayment_df['type'] != 'Manual Discount']

In [36]:
repayment_with_user_id = pd.merge(
    repayment_df[['contract_reference', 'client_id', 'total_amount']],
    client_df[['id', 'user_in_charge_id']],
    left_on='client_id',
    right_on='id',
    how='left'
)

expected_payments = repayment_with_user_id.groupby(
    ['contract_reference', 'user_in_charge_id']
)['total_amount'].sum().reset_index()

reconciled_df['amount'] = reconciled_df['amount'] * -1
reconciled_payments = reconciled_df.groupby(
    'destination_contract_reference'
)['amount'].sum().reset_index()
reconciled_payments

,destination_contract_reference,amount
0,126,175000.0
1,440,175000.0
2,66,95000.0
3,C1000017,490000.0
4,C1010016,455000.0
...,...,...
934,C970012,145000.0
935,C9710013,105000.0
936,C9740010,35000.0
937,C980011,490000.0


In [38]:
reconciled_payments.rename(columns={'destination_contract_reference': 'contract_reference'}, inplace=True)

In [40]:
merged_df = pd.merge(
    expected_payments,
    reconciled_payments,
    on='contract_reference',
    how='left'
).fillna(0)
merged_df.head(10)

merged_df.rename(columns={
    'total_amount': 'expected_amount',
    'amount': 'amount_collected'
}, inplace=True)
merged_df

,contract_reference,user_in_charge_id,expected_amount,amount_collected
0,C1000017,1.0,490000.0,490000.0
1,C1010016,1.0,455000.0,455000.0
2,C1020015,1.0,630000.0,630000.0
3,C1030014,1.0,595000.0,595000.0
4,C1040013,1.0,595000.0,595000.0
...,...,...,...,...
931,C970012,1.0,145000.0,145000.0
932,C9710013,7.0,105000.0,105000.0
933,C9740010,7.0,35000.0,35000.0
934,C980011,1.0,490000.0,490000.0


In [42]:
guardian_performance = merged_df.groupby('user_in_charge_id').agg(
    actual_payments=('amount_collected', 'sum'),
    expected_payments=('expected_amount', 'sum')
).reset_index()

guardian_performance['collection_rate'] = (guardian_performance['actual_payments'] / guardian_performance['expected_payments']) * 100
guardian_performance['collection_rate'] = guardian_performance.apply(
    lambda x: x['collection_rate'] if x['expected_payments'] > 0 else 0,
    axis=1)

guardian_performance['performance_rank'] = guardian_performance['collection_rate'].rank(ascending=False)
guardian_performance.sort_values('collection_rate', ascending=False, inplace=True)
guardian_performance


,user_in_charge_id,actual_payments,expected_payments,collection_rate,performance_rank
1,5.0,5115000.0,5110000.0,100.097847,1.0
0,1.0,112058994.0,112018994.0,100.035708,2.0
3,9.0,50365044.0,50350044.0,100.029791,3.0
2,7.0,155600210.0,155600210.0,100.000000,5.5
4,10.0,3161000.0,3161000.0,100.000000,5.5
5,14.0,17181035.0,17181035.0,100.000000,5.5
6,16.0,24836050.0,24836050.0,100.000000,5.5


### client retention rate

In [45]:
#basic retention rate
#calculate % of clients with active contracts today
total_clients = len(client_df)
active_clients = client_df['has_active_contracts'].sum()

retention_rate = (active_clients / total_clients) * 100
print(f"Current Retention Rate: {retention_rate:.1f}%")

Current Retention Rate: 61.1%


In [47]:
#by user in charge
retention_by_guardian = client_df.groupby('user_in_charge_id')['has_active_contracts'].agg(
    total_clients='count',
    active_clients='sum'
).reset_index()

retention_by_guardian['retention_rate'] = (
    retention_by_guardian['active_clients'] / retention_by_guardian['total_clients']) * 100
retention_by_guardian

,user_in_charge_id,total_clients,active_clients,retention_rate
0,1,265,143,53.962264
1,5,11,5,45.454545
2,7,430,253,58.837209
3,9,121,94,77.685950
4,10,11,5,45.454545
5,14,35,26,74.285714
6,16,59,43,72.881356


In [49]:
#merging with contracts event data file for more in detailed look
contract_events = pd.read_csv('contract_event_data_20250423120141.csv')
contract_events.columns = contract_events.columns.str.lower().str.replace(' ', '_')
contract_events.columns

Index(['id', 'contract_id', 'contract_reference', 'date', 'type',
       'contract_payment_id', 'approved_by_user_id', 'approver_user_name',
       'narration', 'note', 'last_updated'],
      dtype='object')

In [51]:
contract_events['type'].unique()

array(['Undo Default', 'Default', 'Repayment Reversal', 'Manual Discount',
       'Undo Completion', 'Manual Delay', 'Repossession', 'Device Swap',
       'Cancellation', 'Downpayment Reversal', 'Creation', 'Offer Change',
       'Completion'], dtype=object)

In [54]:
#using type of contract event

In [56]:
#retention rate
# Define all event types that indicate non-retention
nonretention_event_types = [
    'Default',
    'Repossession',
    'Cancellation',
    'Downpayment Reversal']

# Filter for these events and get unique contract IDs
nonretain_clients = contract_events[
    contract_events['type'].isin(nonretention_event_types)
]['contract_id'].unique()

#adding defaulted clients into client data
client_df['not_retained'] = client_df['id'].isin(nonretain_clients)

In [58]:
client_df

,id,custom_id,first_name,family_name,birthdate,age,gender,picture_uuid,language_sms,language_spoken,...,l3_entity_id,l3_entity_name,l4_entity_id,l4_entity_name,note,tags,overpaid_amount,user_in_charge_id,last_updated,not_retained
0,933,NaN,Ana,Epinayú Epinayú,NaN,NaN,Female,NaN,NaN,NaN,...,2,Default Zone,1,Default Region,NaN,NaN,0.0,7,2025-04-23 03:00,False
1,932,NaN,Brayan,Prueba,NaN,NaN,Male,NaN,Spanish,NaN,...,2,Default Zone,1,Default Region,NaN,NaN,0.0,1,2025-04-23 03:00,False
2,931,NaN,Maye,Uriana,1990-07-24 00:00,34.0,Female,NaN,NaN,NaN,...,2,Default Zone,1,Default Region,NaN,NaN,0.0,1,2025-04-23 03:00,False
3,930,NaN,Diana Patricia,Uriana Urian,1998-06-15 00:00,26.0,Female,NaN,NaN,NaN,...,2,Default Zone,1,Default Region,NaN,NaN,0.0,1,2025-04-23 03:00,False
4,929,NaN,Rosiris Yosira,Uriana,1999-12-15 00:00,25.0,Female,NaN,NaN,NaN,...,2,Default Zone,1,Default Region,NaN,NaN,0.0,1,2025-04-23 03:00,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
927,5,NaN,Epinayu,Ana Maria,NaN,NaN,Unknown,NaN,NaN,NaN,...,2,Default Zone,1,Default Region,NaN,NaN,0.0,7,2025-04-23 03:00,True
928,4,NaN,Epieyu,Oliverio,NaN,NaN,Unknown,NaN,NaN,NaN,...,2,Default Zone,1,Default Region,NaN,NaN,0.0,7,2025-03-18 17:09,True
929,3,NaN,Epieyu,Uriana Enelisa,NaN,NaN,Unknown,NaN,NaN,NaN,...,2,Default Zone,1,Default Region,NaN,NaN,0.0,7,2025-04-23 03:00,True
930,2,NaN,Paula,Kisabo,NaN,NaN,Unknown,NaN,NaN,NaN,...,2,Default Zone,1,Default Region,NaN,NaN,0.0,1,2024-09-04 15:00,True


In [60]:
#calculate retention
#uses multiple dimensions of retention + captures ppl that completed contracts
retained_clients = client_df[
    (client_df['has_active_contracts']) |
    (~client_df['not_retained'] & client_df['has_completed_contracts'])
].shape[0]

total_clients = client_df.shape[0]
retention_rate = (retained_clients / total_clients) * 100

print(f"Retention Rate: {retention_rate:.1f}%")

Retention Rate: 61.6%


In [62]:
client_df.shape[0]

932

In [64]:
#using not retained column (easier to implement BUT ignores constract completion
retained_clients2 = (client_df['not_retained'] == False).sum()
retention_rate2 = (retained_clients2 / client_df.shape[0]) * 100

In [66]:
retention_rate2

56.75965665236051

In [68]:
#by user in charge
#merge contract events to clients data (via contract id)
contracts_with_guardian = pd.merge(
    contract_events[['contract_id', 'type']],
    client_df[['id', 'user_in_charge_id']],
    left_on='contract_id',
    right_on='id',
    how='left'
)
contracts_with_guardian.head(10)

,contract_id,type,id,user_in_charge_id
0,147,Undo Default,147.0,7.0
1,11,Default,11.0,7.0
2,200,Default,200.0,7.0
3,609,Default,609.0,7.0
4,393,Default,393.0,7.0
5,179,Default,179.0,7.0
6,52,Default,52.0,16.0
7,817,Default,817.0,16.0
8,446,Default,446.0,7.0
9,428,Default,428.0,7.0


In [70]:
contracts_with_guardian[contracts_with_guardian['type'].isin(nonretention_event_types)]['type'].value_counts()

type
Default                 415
Repossession             76
Cancellation              3
Downpayment Reversal      3
Name: count, dtype: int64

In [72]:
# calculate non-retention counts
nonretention_by_guardian = contracts_with_guardian[
    contracts_with_guardian['type'].isin(nonretention_event_types)
].groupby('user_in_charge_id')['contract_id'].nunique().reset_index(name='nonretention_count')
nonretention_by_guardian

,user_in_charge_id,nonretention_count
0,1.0,134
1,5.0,6
2,7.0,196
3,9.0,34
4,10.0,8
5,14.0,9
6,16.0,16


In [74]:
total_contracts_by_guardian = contracts_with_guardian.groupby('user_in_charge_id')['contract_id'].nunique().reset_index(name='total_contracts')

#merge total contracts into nonretention by guardian
nonretention_by_guardian = nonretention_by_guardian.merge(
    total_contracts_by_guardian,
    on='user_in_charge_id',
    how='left'
)
nonretention_by_guardian

,user_in_charge_id,nonretention_count,total_contracts
0,1.0,134,265
1,5.0,6,11
2,7.0,196,430
3,9.0,34,121
4,10.0,8,11
5,14.0,9,35
6,16.0,16,59


In [76]:
nonretention_by_guardian['nonretention_rate'] = (
    nonretention_by_guardian['nonretention_count'] /
    nonretention_by_guardian['total_contracts']
) * 100
nonretention_by_guardian['retention_rate'] = 100 - nonretention_by_guardian['nonretention_rate']
nonretention_by_guardian = nonretention_by_guardian.sort_values(
    by='retention_rate', 
    ascending=False
).reset_index(drop=True)
nonretention_by_guardian

,user_in_charge_id,nonretention_count,total_contracts,nonretention_rate,retention_rate
0,14.0,9,35,25.714286,74.285714
1,16.0,16,59,27.118644,72.881356
2,9.0,34,121,28.099174,71.900826
3,7.0,196,430,45.581395,54.418605
4,1.0,134,265,50.566038,49.433962
5,5.0,6,11,54.545455,45.454545
6,10.0,8,11,72.727273,27.272727


In [78]:
user_df = pd.read_csv('user_data_20250423120106.csv')
user_df.columns = user_df.columns.str.lower().str.replace(' ', '_')
print(user_df.columns)

Index(['id', 'first_name', 'family_name', 'email', 'username', 'role',
       'gender', 'reference_entity_id', 'reference_entity_name',
       'primary_phone_number', 'all_phone_numbers', 'last_web_access',
       'last_mobile_sync', 'last_mobile_app_version', 'expiration_date',
       'last_updated'],
      dtype='object')


In [80]:
# Merge with user_df to add role information
nonretention_with_roles = nonretention_by_guardian.merge(
    user_df[['id', 'role']],  
    left_on='user_in_charge_id',
    right_on='id',
    how='left'
).drop(columns='id')  

# Reorder columns for better readability
column_order = [
    'user_in_charge_id', 
    'role',
    'nonretention_count', 
    'total_contracts',
    'nonretention_rate',
    'retention_rate']
nonretention_with_roles = nonretention_with_roles[column_order]
nonretention_with_roles

,user_in_charge_id,role,nonretention_count,total_contracts,nonretention_rate,retention_rate
0,14.0,Agent,9,35,25.714286,74.285714
1,16.0,Agent,16,59,27.118644,72.881356
2,9.0,Agent,34,121,28.099174,71.900826
3,7.0,Admin,196,430,45.581395,54.418605
4,1.0,SuperAdmin,134,265,50.566038,49.433962
5,5.0,ViewOnly,6,11,54.545455,45.454545
6,10.0,Agent,8,11,72.727273,27.272727


### average payment delays

In [83]:
repayment_df['payment_date'] = pd.to_datetime(repayment_df['payment_date']).dt.date
repayment_df['expected_payment_date'] = pd.to_datetime(repayment_df['expected_payment_date']).dt.date
repayment_df

,id,contract_id,contract_reference,client_id,payment_date,expected_payment_date,days_late,total_amount,amount_paid,amount_of_discount,credit_value,type,last_updated
0,8542,712,C7910011,712.0,2025-04-23,2025-04-22,1.01,35000.0,35000.0,0.0,35.0,NaN,2025-04-23 01:54
1,8541,71,C890012,71.0,2025-04-23,2025-04-11,12.01,30000.0,30000.0,0.0,30.0,NaN,2025-04-23 01:09
2,8540,812,C8920019,812.0,2025-04-23,2025-03-29,24.07,35000.0,35000.0,0.0,30.0,NaN,2025-04-23 00:54
3,8539,230,C2820017,230.0,2025-04-23,2025-04-20,2.02,35000.0,35000.0,0.0,30.0,NaN,2025-04-23 00:06
4,8538,134,C1860014,134.0,2025-04-22,2025-04-23,-0.05,35000.0,35000.0,0.0,30.0,NaN,2025-04-22 23:43
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8537,5,4,C220012,4.0,2024-07-11,2024-07-11,NaN,35000.0,35000.0,0.0,30.0,Downpayment,2024-07-11 16:11
8538,4,3,C210013,3.0,2024-07-11,2024-08-10,-30.00,210000.0,210000.0,0.0,180.0,NaN,2024-07-11 16:11
8539,3,3,C210013,3.0,2024-07-11,2024-07-11,NaN,35000.0,35000.0,0.0,30.0,Downpayment,2024-07-11 16:11
8540,2,2,C90019,2.0,2024-06-20,2024-06-20,NaN,30000.0,30000.0,0.0,30.0,Downpayment,2024-06-20 16:27


In [85]:
#using days late directly
#calculate average delay
avg_delay = repayment_df['days_late'].mean()
print(f"Overall Average Delay: {avg_delay:.1f} days")

Overall Average Delay: -67.3 days


In [87]:
#by user in charge
merged_df2 = pd.merge(
    repayment_df,
    client_df[['id', 'user_in_charge_id']],
    left_on='client_id',
    right_on='id',
    how='left'
)

delay_by_guardian = merged_df2.groupby('user_in_charge_id').agg(
    avg_delay=('days_late', 'mean'),
    median_delay=('days_late', 'median'),
    late_payments=('days_late', lambda x: (x > 0).sum()),
    total_payments=('days_late', 'count')
).reset_index()

print(delay_by_guardian.sort_values('avg_delay'))

   user_in_charge_id   avg_delay  median_delay  late_payments  total_payments
2                7.0 -112.727009         0.040           1665            3256
1                5.0  -64.946290         0.725             33              62
0                1.0  -40.820688        -2.020            805            1875
5               14.0  -29.914739        -0.025            209             422
6               16.0  -28.692660       -11.985            228             624
4               10.0  -21.303115         0.740             33              61
3                9.0  -21.151853        -0.025            591            1198


In [89]:
#removing rows where amount paid is 0 or in negatives (discounts & payment reversals)
repayment_copy = repayment_df.copy()
repayment_copy_filtered = repayment_copy[repayment_copy['amount_paid'] > 0].copy()
repayment_copy_filtered['payment_date'] = pd.to_datetime(repayment_copy_filtered['payment_date']).dt.date
repayment_copy_filtered['expected_payment_date'] = pd.to_datetime(repayment_copy_filtered['expected_payment_date']).dt.date

In [91]:
#average days late
avg_delay = repayment_copy_filtered['days_late'].mean()
print(f"Overall Average Delay: {avg_delay:.1f} days")

#by user in charge
merged_df3 = pd.merge(
    repayment_copy_filtered,
    client_df[['id', 'user_in_charge_id']],
    left_on='client_id',
    right_on='id',
    how='left'
)

delay_by_guardian = merged_df3.groupby('user_in_charge_id').agg(
    avg_delay=('days_late', 'mean'),
    median_delay=('days_late', 'median'),
    late_payments=('days_late', lambda x: (x > 0).sum()),
    total_payments=('days_late', 'count')
).reset_index()

delay_by_guardian['late_payment_percent'] = (
    delay_by_guardian['late_payments'] / delay_by_guardian['total_payments'] * 100
).round(2)
delay_by_guardian = delay_by_guardian.sort_values(
    by='late_payment_percent', 
    ascending=True
).reset_index(drop=True)
delay_by_guardian

Overall Average Delay: -13.6 days


,user_in_charge_id,avg_delay,median_delay,late_payments,total_payments,late_payment_percent
0,16,-77.849251,-29.995,124,414,29.95
1,1,-15.057729,-0.010,668,1343,49.74
2,14,-21.351465,0.860,165,273,60.44
3,9,-2.169602,0.980,509,830,61.33
4,7,-5.419303,1.035,1481,2366,62.60
5,5,-0.583830,2.150,31,47,65.96
6,10,11.429091,2.870,33,44,75.00


In [93]:
# Merge with user_df to add role information
delay_by_guardian_with_roles = delay_by_guardian.merge(
    user_df[['id', 'role']],  
    left_on='user_in_charge_id',
    right_on='id',
    how='left'
).drop(columns='id')  

# Reorder columns for better readability
column_order = [
    'user_in_charge_id', 
    'role',
    'avg_delay', 
    'median_delay',
    'total_payments',
    'late_payments',
    'late_payment_percent']
delay_by_guardian_with_roles = delay_by_guardian_with_roles[column_order]
delay_by_guardian_with_roles

,user_in_charge_id,role,avg_delay,median_delay,total_payments,late_payments,late_payment_percent
0,16,Agent,-77.849251,-29.995,414,124,29.95
1,1,SuperAdmin,-15.057729,-0.010,1343,668,49.74
2,14,Agent,-21.351465,0.860,273,165,60.44
3,9,Agent,-2.169602,0.980,830,509,61.33
4,7,Admin,-5.419303,1.035,2366,1481,62.60
5,5,ViewOnly,-0.583830,2.150,47,31,65.96
6,10,Agent,11.429091,2.870,44,33,75.00


In [95]:
#calculating days late from payment date - expected (verify days late)
repayment_copy_filtered['payment_date'] = pd.to_datetime(repayment_copy_filtered['payment_date'])
repayment_copy_filtered['expected_payment_date'] = pd.to_datetime(repayment_copy_filtered['expected_payment_date'])
repayment_copy_filtered['calculated_delay'] = (
    (repayment_copy_filtered['payment_date'] - repayment_copy_filtered['expected_payment_date']).dt.days)
print(repayment_copy_filtered[['days_late', 'calculated_delay']].head(10))

   days_late  calculated_delay
0       1.01                 1
1      12.01                12
2      24.07                25
3       2.02                 3
4      -0.05                -1
5       1.05                 1
6       9.55                 9
7      -2.05                -3
8      52.32                52
9      56.19                56


In [97]:
days_late_accuracy = (
    (repayment_copy_filtered['calculated_delay'] == repayment_copy_filtered['days_late'].round()).mean() * 100
)
print(f"Match rate: {days_late_accuracy:.2f}%")
#previous 52%

Match rate: 67.02%


In [99]:
#average days late using calculations
avg_delay = repayment_copy_filtered['calculated_delay'].mean()
print(f"Overall Average Delay: {avg_delay:.1f} days")

Overall Average Delay: -11.6 days


In [132]:
contract_events['type'].unique()

array(['Undo Default', 'Default', 'Repayment Reversal', 'Manual Discount',
       'Undo Completion', 'Manual Delay', 'Repossession', 'Device Swap',
       'Cancellation', 'Downpayment Reversal', 'Creation', 'Offer Change',
       'Completion'], dtype=object)

In [134]:
contract_events = pd.read_csv('contract_event_data_20250423120141.csv')
contract_events.columns = contract_events.columns.str.lower().str.replace(' ', '_')
contract_events.columns

Index(['id', 'contract_id', 'contract_reference', 'date', 'type',
       'contract_payment_id', 'approved_by_user_id', 'approver_user_name',
       'narration', 'note', 'last_updated'],
      dtype='object')

### leads generated by guardians

In [137]:
leads_df = pd.read_csv('lead_data_20250423120123.csv')
leads_gen_df = pd.read_csv('lead_generator_data_20250423120127.csv')
user_df = pd.read_csv('user_data_20250423120106.csv')

#standardize column headers
leads_df.columns = leads_df.columns.str.lower().str.replace(' ', '_')
leads_gen_df.columns = leads_gen_df.columns.str.lower().str.replace(' ', '_')
user_df.columns = user_df.columns.str.lower().str.replace(' ', '_')
print(leads_df.columns)
print(leads_gen_df.columns)
print(user_df.columns)

Index(['id', 'custom_id', 'first_name', 'family_name', 'birthdate', 'age',
       'gender', 'picture_uuid', 'language_sms', 'language_spoken',
       'longitude', 'latitude', 'primary_phone_number', 'all_phone_numbers',
       'status', 'note', 'reasons_for_not_buying', 'commission', 'offer_id',
       'offer_name', 'lead_generator_id', 'lead_generator_name', 'entry_date',
       'generation_date', 'next_contact_date', 'planned_delivery_date',
       'last_status_change_date', 'decision_date', 'decision_by_user_id',
       'decision_user_name', 'client_id', 'contract_reference', 'portfolio_id',
       'portfolio_name', 'client_group_id', 'client_group_name',
       'l0_entity_id', 'l0_entity_name', 'l1_entity_id', 'l1_entity_name',
       'l2_entity_id', 'l2_entity_name', 'l3_entity_id', 'l3_entity_name',
       'l4_entity_id', 'l4_entity_name', 'user_in_charge_id', 'last_updated'],
      dtype='object')
Index(['id', 'first_name', 'family_name', 'primary_phone_number',
       'all_phon

In [139]:
#based on contract reference (non-null contract referencer = converted) 
leads_per_generator = leads_df.groupby(
    ['lead_generator_id', 'lead_generator_name']
).agg(
    total_leads=('id', 'count'),
    converted_leads=('contract_reference', lambda x: x.notna().sum())
).reset_index()
leads_per_generator

,lead_generator_id,lead_generator_name,total_leads,converted_leads
0,1,Default Lead Generator,2,2
1,2,Cristina Epiayú,169,169
2,3,Gabriel Epieyú,48,48
3,4,Gisela Ipuana,114,114
4,5,Alder Epiayú,89,89
5,6,Juan Pablo Epiayú,97,97
6,7,Jaider Bouriyú,54,54
7,8,Luis Castellanos,103,103
8,9,Nurys Ballesteros,38,38
9,10,Ricardo Mejía,38,38


In [141]:
leads_df['status'].unique()

array(['Installed', 'Hesitating', 'Discarded', 'Cancelled',
       'Ready to Buy'], dtype=object)

In [143]:
#based on status: installed & ready to buy
leads_per_generator_status = leads_df.groupby(
    ['lead_generator_id', 'lead_generator_name']
).agg(
    total_leads=('id', 'count'),
    converted_leads=('status', lambda x: x.isin(['Installed', 'Ready to Buy']).sum())
).reset_index()
leads_per_generator_status['conversion_rate'] = (
    leads_per_generator_status['converted_leads'] / leads_per_generator_status['total_leads']) * 100
leads_per_generator_status = leads_per_generator_status.sort_values('conversion_rate', ascending=False)
leads_per_generator_status

,lead_generator_id,lead_generator_name,total_leads,converted_leads,conversion_rate
0,1,Default Lead Generator,2,2,100.000000
8,9,Nurys Ballesteros,38,38,100.000000
16,20,Emilio Urariyu,28,28,100.000000
15,17,Alvaro Epiayú,35,35,100.000000
14,16,Libardo Epiayú,24,24,100.000000
13,14,Darianna Epiayú,23,23,100.000000
12,13,José Miguel Malo,1,1,100.000000
10,11,Rita Vanegas,10,10,100.000000
9,10,Ricardo Mejía,38,38,100.000000
6,7,Jaider Bouriyú,54,54,100.000000


In [145]:
# Merge with user_df to add role information
leads_per_generator_status = leads_per_generator_status.merge(
    user_df[['id', 'role']],  
    left_on='lead_generator_id',
    right_on='id',
    how='left'
).drop(columns='id')  

# Reorder columns for better readability
column_order = [
    'lead_generator_id', 
    'lead_generator_name',
    'role', 
    'total_leads',
    'converted_leads',
    'conversion_rate']
leads_per_generator_status = leads_per_generator_status[column_order]
leads_per_generator_status

,lead_generator_id,lead_generator_name,role,total_leads,converted_leads,conversion_rate
0,1,Default Lead Generator,SuperAdmin,2,2,100.000000
1,9,Nurys Ballesteros,Agent,38,38,100.000000
2,20,Emilio Urariyu,ViewOnly,28,28,100.000000
3,17,Alvaro Epiayú,ViewOnly,35,35,100.000000
4,16,Libardo Epiayú,Agent,24,24,100.000000
5,14,Darianna Epiayú,Agent,23,23,100.000000
6,13,José Miguel Malo,Agent,1,1,100.000000
7,11,Rita Vanegas,Agent,10,10,100.000000
8,10,Ricardo Mejía,Agent,38,38,100.000000
9,7,Jaider Bouriyú,Admin,54,54,100.000000


### merge collection rate with payment delays

In [152]:
guardian_kpis = pd.merge(
    delay_by_guardian_with_roles,
    guardian_performance[['user_in_charge_id', 'collection_rate']],
    on='user_in_charge_id',
    how='left'
)
guardian_kpis

,user_in_charge_id,role,avg_delay,median_delay,total_payments,late_payments,late_payment_percent,collection_rate
0,16,Agent,-77.849251,-29.995,414,124,29.95,100.000000
1,1,SuperAdmin,-15.057729,-0.010,1343,668,49.74,100.035708
2,14,Agent,-21.351465,0.860,273,165,60.44,100.000000
3,9,Agent,-2.169602,0.980,830,509,61.33,100.029791
4,7,Admin,-5.419303,1.035,2366,1481,62.60,100.000000
5,5,ViewOnly,-0.583830,2.150,47,31,65.96,100.097847
6,10,Agent,11.429091,2.870,44,33,75.00,100.000000


### merge with client retention rate

In [158]:
guardian_kpis2 = pd.merge(
    guardian_kpis,
    nonretention_with_roles[['user_in_charge_id', 'retention_rate']],
    on='user_in_charge_id',
    how='left'
)
guardian_kpis2

,user_in_charge_id,role,avg_delay,median_delay,total_payments,late_payments,late_payment_percent,collection_rate,retention_rate
0,16,Agent,-77.849251,-29.995,414,124,29.95,100.000000,72.881356
1,1,SuperAdmin,-15.057729,-0.010,1343,668,49.74,100.035708,49.433962
2,14,Agent,-21.351465,0.860,273,165,60.44,100.000000,74.285714
3,9,Agent,-2.169602,0.980,830,509,61.33,100.029791,71.900826
4,7,Admin,-5.419303,1.035,2366,1481,62.60,100.000000,54.418605
5,5,ViewOnly,-0.583830,2.150,47,31,65.96,100.097847,45.454545
6,10,Agent,11.429091,2.870,44,33,75.00,100.000000,27.272727


In [160]:
guardian_kpis2 = guardian_kpis2.drop(columns=[
    'avg_delay', 'median_delay', 'total_payments', 'late_payments'
])
guardian_kpis2

,user_in_charge_id,role,late_payment_percent,collection_rate,retention_rate
0,16,Agent,29.95,100.000000,72.881356
1,1,SuperAdmin,49.74,100.035708,49.433962
2,14,Agent,60.44,100.000000,74.285714
3,9,Agent,61.33,100.029791,71.900826
4,7,Admin,62.60,100.000000,54.418605
5,5,ViewOnly,65.96,100.097847,45.454545
6,10,Agent,75.00,100.000000,27.272727


In [162]:
guardian_kpis3 = pd.merge(
    guardian_kpis2,
    leads_per_generator_status[['lead_generator_id', 'conversion_rate']],
    left_on='user_in_charge_id',
    right_on='lead_generator_id',
    how='left'
).drop(columns='lead_generator_id')
guardian_kpis3

,user_in_charge_id,role,late_payment_percent,collection_rate,retention_rate,conversion_rate
0,16,Agent,29.95,100.000000,72.881356,100.0
1,1,SuperAdmin,49.74,100.035708,49.433962,100.0
2,14,Agent,60.44,100.000000,74.285714,100.0
3,9,Agent,61.33,100.029791,71.900826,100.0
4,7,Admin,62.60,100.000000,54.418605,100.0
5,5,ViewOnly,65.96,100.097847,45.454545,100.0
6,10,Agent,75.00,100.000000,27.272727,100.0


In [147]:
#drop client id from rec because empty
reconciled_df = reconciled_df.drop(columns=['client_id'])
reconciled_df

,id,date,amount,type,note,payment_wallet_name,destination_type,destination_contract_reference,origin_payment_id,origin_payment_transaction_id,payment_wallet_id,lead_id,add_on_id,contract_payment_id,contract_pending_payment_id,user_id,last_updated
0,7388,2025-04-23 01:54,35000.0,Contract Payment,NaN,Neila Ipuana (L:774) [Cash],Contract Payment,C7910011,10909.0,05b7-3600,782,NaN,NaN,8542.0,NaN,NaN,2025-04-23 01:54
1,7387,2025-04-23 01:09,30000.0,Contract Payment,NaN,Alirio Barliza Mengual (L:72) [Cash],Contract Payment,C890012,10907.0,2bca-d1ec,83,NaN,NaN,8541.0,NaN,NaN,2025-04-23 01:09
2,7386,2025-04-23 00:54,35000.0,Contract Payment,NaN,Yaidy Jusayu (L:875) [Cash],Contract Payment,C8920019,10905.0,eed4-d2e9,881,NaN,NaN,8540.0,NaN,NaN,2025-04-23 00:54
3,7385,2025-04-23 00:06,35000.0,Contract Payment,NaN,Ana Josefina Epieyu Pushaina (L:265) [Cash],Contract Payment,C2820017,10903.0,17cc-b016,275,NaN,NaN,8539.0,NaN,NaN,2025-04-23 00:06
4,7384,2025-04-22 23:43,35000.0,Contract Payment,NaN,Mileidis Maria Epieyu Epieyu (L:169) [Cash],Contract Payment,C1860014,10901.0,d898-7f59,180,NaN,NaN,8538.0,NaN,NaN,2025-04-22 23:43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,5,2024-07-11 08:24,35000.0,Downpayment,NaN,Epieyu Oliverio (L:5) [Cash],Contract Payment,C220012,7.0,9455-fd56,16,5.0,NaN,5.0,NaN,NaN,2024-07-11 16:11
7008,4,2024-07-11 08:24,210000.0,Contract Payment,NaN,Epieyu Uriana Enelisa (L:4) [Cash],Contract Payment,C210013,5.0,9451-80b1,15,4.0,NaN,4.0,NaN,NaN,2024-07-11 16:11
7009,3,2024-07-11 08:24,35000.0,Downpayment,NaN,Epieyu Uriana Enelisa (L:4) [Cash],Contract Payment,C210013,5.0,9451-80b1,15,4.0,NaN,3.0,NaN,NaN,2024-07-11 16:11
7010,2,2024-06-20 16:25,30000.0,Downpayment,NaN,Paula Kisabo (L:2) [Cash],Contract Payment,C90019,3.0,228b-a93a,5,2.0,NaN,2.0,NaN,NaN,2024-06-20 16:27


In [ ]:
print(reconciled_df['destination_contract_reference'].value_counts().head())
print(repayment_df['contract_reference'].value_counts().head())


destination_contract_reference
C5750013    45
C620013     42
C550012     40
C7620016    36
C7960016    32
Name: count, dtype: int64
contract_reference
C5750013    51
C620013     46
C550012     44
C7620016    37
C5820014    36
Name: count, dtype: int64


In [ ]:
repayment_deduped = repayment_df[['contract_reference', 'client_id']].drop_duplicates(subset='contract_reference')

In [ ]:
#add client id to reconcile from repayment
recon_w_clientid = reconciled_df.merge(
    repayment_deduped,
    left_on='destination_contract_reference',
    right_on='contract_reference',
    how='left')
recon_w_clientid

,id,date,amount,type,note,payment_wallet_name,destination_type,destination_contract_reference,origin_payment_id,origin_payment_transaction_id,payment_wallet_id,lead_id,add_on_id,contract_payment_id,contract_pending_payment_id,user_id,last_updated,contract_reference,client_id
0,7388,2025-04-23 01:54:00,35000.0,Contract Payment,NaN,Neila Ipuana (L:774) [Cash],Contract Payment,C7910011,10909.0,05b7-3600,782,NaN,NaN,8542.0,NaN,NaN,2025-04-23 01:54,C7910011,712.0
1,7387,2025-04-23 01:09:00,30000.0,Contract Payment,NaN,Alirio Barliza Mengual (L:72) [Cash],Contract Payment,C890012,10907.0,2bca-d1ec,83,NaN,NaN,8541.0,NaN,NaN,2025-04-23 01:09,C890012,71.0
2,7386,2025-04-23 00:54:00,35000.0,Contract Payment,NaN,Yaidy Jusayu (L:875) [Cash],Contract Payment,C8920019,10905.0,eed4-d2e9,881,NaN,NaN,8540.0,NaN,NaN,2025-04-23 00:54,C8920019,812.0
3,7385,2025-04-23 00:06:00,35000.0,Contract Payment,NaN,Ana Josefina Epieyu Pushaina (L:265) [Cash],Contract Payment,C2820017,10903.0,17cc-b016,275,NaN,NaN,8539.0,NaN,NaN,2025-04-23 00:06,C2820017,230.0
4,7384,2025-04-22 23:43:00,35000.0,Contract Payment,NaN,Mileidis Maria Epieyu Epieyu (L:169) [Cash],Contract Payment,C1860014,10901.0,d898-7f59,180,NaN,NaN,8538.0,NaN,NaN,2025-04-22 23:43,C1860014,134.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,5,2024-07-11 08:24:00,35000.0,Downpayment,NaN,Epieyu Oliverio (L:5) [Cash],Contract Payment,C220012,7.0,9455-fd56,16,5.0,NaN,5.0,NaN,NaN,2024-07-11 16:11,C220012,4.0
7008,4,2024-07-11 08:24:00,210000.0,Contract Payment,NaN,Epieyu Uriana Enelisa (L:4) [Cash],Contract Payment,C210013,5.0,9451-80b1,15,4.0,NaN,4.0,NaN,NaN,2024-07-11 16:11,C210013,3.0
7009,3,2024-07-11 08:24:00,35000.0,Downpayment,NaN,Epieyu Uriana Enelisa (L:4) [Cash],Contract Payment,C210013,5.0,9451-80b1,15,4.0,NaN,3.0,NaN,NaN,2024-07-11 16:11,C210013,3.0
7010,2,2024-06-20 16:25:00,30000.0,Downpayment,NaN,Paula Kisabo (L:2) [Cash],Contract Payment,C90019,3.0,228b-a93a,5,2.0,NaN,2.0,NaN,NaN,2024-06-20 16:27,C90019,2.0


In [ ]:
recon_w_clientid.columns

Index(['id', 'date', 'amount', 'type', 'note', 'payment_wallet_name',
       'destination_type', 'destination_contract_reference',
       'origin_payment_id', 'origin_payment_transaction_id',
       'payment_wallet_id', 'lead_id', 'add_on_id', 'contract_payment_id',
       'contract_pending_payment_id', 'user_id', 'last_updated',
       'contract_reference', 'client_id'],
      dtype='object')

In [ ]:
#add villages to recon with client id
rec_w_village = recon_w_clientid.merge(
    client_df[['id', 'l0_entity_name', 'user_in_charge_id']],
    left_on='client_id',
    right_on='id',
    how='left')
rec_w_village = rec_w_village.drop(columns='id_y')
rec_w_village

,id_x,date,amount,type,note,payment_wallet_name,destination_type,destination_contract_reference,origin_payment_id,origin_payment_transaction_id,...,lead_id,add_on_id,contract_payment_id,contract_pending_payment_id,user_id,last_updated,contract_reference,client_id,l0_entity_name,user_in_charge_id
0,7388,2025-04-23 01:54:00,35000.0,Contract Payment,NaN,Neila Ipuana (L:774) [Cash],Contract Payment,C7910011,10909.0,05b7-3600,...,NaN,NaN,8542.0,NaN,NaN,2025-04-23 01:54,C7910011,712.0,Waimpretru,9.0
1,7387,2025-04-23 01:09:00,30000.0,Contract Payment,NaN,Alirio Barliza Mengual (L:72) [Cash],Contract Payment,C890012,10907.0,2bca-d1ec,...,NaN,NaN,8541.0,NaN,NaN,2025-04-23 01:09,C890012,71.0,Jasaishao,5.0
2,7386,2025-04-23 00:54:00,35000.0,Contract Payment,NaN,Yaidy Jusayu (L:875) [Cash],Contract Payment,C8920019,10905.0,eed4-d2e9,...,NaN,NaN,8540.0,NaN,NaN,2025-04-23 00:54,C8920019,812.0,Waimpretru,9.0
3,7385,2025-04-23 00:06:00,35000.0,Contract Payment,NaN,Ana Josefina Epieyu Pushaina (L:265) [Cash],Contract Payment,C2820017,10903.0,17cc-b016,...,NaN,NaN,8539.0,NaN,NaN,2025-04-23 00:06,C2820017,230.0,Kainnamana,7.0
4,7384,2025-04-22 23:43:00,35000.0,Contract Payment,NaN,Mileidis Maria Epieyu Epieyu (L:169) [Cash],Contract Payment,C1860014,10901.0,d898-7f59,...,NaN,NaN,8538.0,NaN,NaN,2025-04-22 23:43,C1860014,134.0,Kainnamana,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,5,2024-07-11 08:24:00,35000.0,Downpayment,NaN,Epieyu Oliverio (L:5) [Cash],Contract Payment,C220012,7.0,9455-fd56,...,5.0,NaN,5.0,NaN,NaN,2024-07-11 16:11,C220012,4.0,Karapashen,7.0
7008,4,2024-07-11 08:24:00,210000.0,Contract Payment,NaN,Epieyu Uriana Enelisa (L:4) [Cash],Contract Payment,C210013,5.0,9451-80b1,...,4.0,NaN,4.0,NaN,NaN,2024-07-11 16:11,C210013,3.0,Alanachikii,7.0
7009,3,2024-07-11 08:24:00,35000.0,Downpayment,NaN,Epieyu Uriana Enelisa (L:4) [Cash],Contract Payment,C210013,5.0,9451-80b1,...,4.0,NaN,3.0,NaN,NaN,2024-07-11 16:11,C210013,3.0,Alanachikii,7.0
7010,2,2024-06-20 16:25:00,30000.0,Downpayment,NaN,Paula Kisabo (L:2) [Cash],Contract Payment,C90019,3.0,228b-a93a,...,2.0,NaN,2.0,NaN,NaN,2024-06-20 16:27,C90019,2.0,Default Village,1.0


In [ ]:
# Clean and prepare data
rec_w_village.rename(columns={'l0_entity_name': 'village', 'amount': 'revenue'}, inplace=True)
rec_w_village['date'] = pd.to_datetime(rec_w_village['date'])

# Convert revenue to positive values (assuming negative amounts represent payments)
rec_w_village['revenue'] = rec_w_village['revenue']* (-1)
rec_w_village

,id_x,date,revenue,type,note,payment_wallet_name,destination_type,destination_contract_reference,origin_payment_id,origin_payment_transaction_id,...,lead_id,add_on_id,contract_payment_id,contract_pending_payment_id,user_id,last_updated,contract_reference,client_id,village,user_in_charge_id
0,7388,2025-04-23 01:54:00,35000.0,Contract Payment,NaN,Neila Ipuana (L:774) [Cash],Contract Payment,C7910011,10909.0,05b7-3600,...,NaN,NaN,8542.0,NaN,NaN,2025-04-23 01:54,C7910011,712.0,Waimpretru,9.0
1,7387,2025-04-23 01:09:00,30000.0,Contract Payment,NaN,Alirio Barliza Mengual (L:72) [Cash],Contract Payment,C890012,10907.0,2bca-d1ec,...,NaN,NaN,8541.0,NaN,NaN,2025-04-23 01:09,C890012,71.0,Jasaishao,5.0
2,7386,2025-04-23 00:54:00,35000.0,Contract Payment,NaN,Yaidy Jusayu (L:875) [Cash],Contract Payment,C8920019,10905.0,eed4-d2e9,...,NaN,NaN,8540.0,NaN,NaN,2025-04-23 00:54,C8920019,812.0,Waimpretru,9.0
3,7385,2025-04-23 00:06:00,35000.0,Contract Payment,NaN,Ana Josefina Epieyu Pushaina (L:265) [Cash],Contract Payment,C2820017,10903.0,17cc-b016,...,NaN,NaN,8539.0,NaN,NaN,2025-04-23 00:06,C2820017,230.0,Kainnamana,7.0
4,7384,2025-04-22 23:43:00,35000.0,Contract Payment,NaN,Mileidis Maria Epieyu Epieyu (L:169) [Cash],Contract Payment,C1860014,10901.0,d898-7f59,...,NaN,NaN,8538.0,NaN,NaN,2025-04-22 23:43,C1860014,134.0,Kainnamana,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,5,2024-07-11 08:24:00,35000.0,Downpayment,NaN,Epieyu Oliverio (L:5) [Cash],Contract Payment,C220012,7.0,9455-fd56,...,5.0,NaN,5.0,NaN,NaN,2024-07-11 16:11,C220012,4.0,Karapashen,7.0
7008,4,2024-07-11 08:24:00,210000.0,Contract Payment,NaN,Epieyu Uriana Enelisa (L:4) [Cash],Contract Payment,C210013,5.0,9451-80b1,...,4.0,NaN,4.0,NaN,NaN,2024-07-11 16:11,C210013,3.0,Alanachikii,7.0
7009,3,2024-07-11 08:24:00,35000.0,Downpayment,NaN,Epieyu Uriana Enelisa (L:4) [Cash],Contract Payment,C210013,5.0,9451-80b1,...,4.0,NaN,3.0,NaN,NaN,2024-07-11 16:11,C210013,3.0,Alanachikii,7.0
7010,2,2024-06-20 16:25:00,30000.0,Downpayment,NaN,Paula Kisabo (L:2) [Cash],Contract Payment,C90019,3.0,228b-a93a,...,2.0,NaN,2.0,NaN,NaN,2024-06-20 16:27,C90019,2.0,Default Village,1.0


In [ ]:
# Calculate total revenue by village and rank
village_revenue = rec_w_village.groupby('village', as_index=False)['revenue'].sum()
village_revenue['rank'] = village_revenue['revenue'].rank(ascending=False, method='min')
village_revenue

,village,revenue,rank
0,Alanachikii,700000.0,30.0
1,Amaichon,4578000.0,17.0
2,Atakaralu,1925000.0,24.0
3,Bukuamake,14825000.0,8.0
4,Default Village,95000.0,32.0
5,El Colorado,17181035.0,7.0
6,Jasainapu,1785000.0,25.0
7,Jasaishao,5115000.0,16.0
8,Joco Mau,1425110.0,27.0
9,Jonchon,13020000.0,10.0


In [ ]:
village_revenue_ranked = village_revenue.sort_values(by='rank', ascending=True).reset_index(drop=True)
village_revenue_ranked

,village,revenue,rank
0,Kainnamana,63489500.0,1.0
1,Waimpretru,46165044.0,2.0
2,Maniature,33395500.0,3.0
3,Uyaraica,29114140.0,4.0
4,Kayetamana,27428109.0,5.0
5,Sabanatico,17467000.0,6.0
6,El Colorado,17181035.0,7.0
7,Bukuamake,14825000.0,8.0
8,Kushaumake,14270000.0,9.0
9,Jonchon,13020000.0,10.0


In [ ]:
# Group by both village and Guardian, then sum revenue
village_guardian_revenue = (
    rec_w_village.groupby(
        ['village', 'user_in_charge_id'],
        as_index=False).agg(
        total_revenue=('revenue', 'sum'),
        payment_count=('revenue', 'count'))
    .sort_values('total_revenue', ascending=False))

In [ ]:
village_guardian_revenue

,village,user_in_charge_id,total_revenue,payment_count
12,Kainnamana,7.0,63489500.0,1382
29,Waimpretru,9.0,46165044.0,909
19,Maniature,7.0,33395500.0,616
27,Uyaraica,7.0,29114140.0,485
15,Kayetamana,1.0,27428109.0,369
23,Sabanatico,1.0,17467000.0,249
5,El Colorado,14.0,17181035.0,341
3,Bukuamake,16.0,14825000.0,347
16,Kushaumake,1.0,14270000.0,173
9,Jonchon,1.0,13020000.0,249


In [ ]:
#top user by villages
# Group by village and user_id, then sum revenue
village_user_revenue = rec_w_village.groupby(
    ['village', 'user_in_charge_id']
)['revenue'].sum().reset_index()

# Rank users within each village by revenue
village_user_revenue['rank'] = village_user_revenue.groupby('village')['revenue'].rank(
    ascending=False,
    method='min')
village_user_revenue

,village,user_in_charge_id,revenue,rank
0,Alanachikii,7.0,700000.0,1.0
1,Amaichon,7.0,4578000.0,1.0
2,Atakaralu,7.0,1925000.0,1.0
3,Bukuamake,16.0,14825000.0,1.0
4,Default Village,1.0,95000.0,1.0
5,El Colorado,14.0,17181035.0,1.0
6,Jasainapu,1.0,1785000.0,1.0
7,Jasaishao,5.0,5115000.0,1.0
8,Joco Mau,1.0,1425110.0,1.0
9,Jonchon,1.0,13020000.0,1.0


In [ ]:
print(rec_w_village.head())
print(rec_w_village.columns)

   id_x                date  revenue              type note  \
0  7388 2025-04-23 01:54:00  35000.0  Contract Payment  NaN   
1  7387 2025-04-23 01:09:00  30000.0  Contract Payment  NaN   
2  7386 2025-04-23 00:54:00  35000.0  Contract Payment  NaN   
3  7385 2025-04-23 00:06:00  35000.0  Contract Payment  NaN   
4  7384 2025-04-22 23:43:00  35000.0  Contract Payment  NaN   

                           payment_wallet_name  destination_type  \
0                  Neila Ipuana (L:774) [Cash]  Contract Payment   
1         Alirio Barliza Mengual (L:72) [Cash]  Contract Payment   
2                  Yaidy Jusayu (L:875) [Cash]  Contract Payment   
3  Ana Josefina Epieyu Pushaina (L:265) [Cash]  Contract Payment   
4  Mileidis Maria Epieyu Epieyu (L:169) [Cash]  Contract Payment   

  destination_contract_reference  origin_payment_id  \
0                       C7910011            10909.0   
1                        C890012            10907.0   
2                       C8920019            109

In [ ]:
#Rank Villages based on total revenue per contract
num_village_contracts = rec_w_village.groupby('Village', as_index = False)['Contract Reference'].nunique().rename(columns = {'Contract Reference': 'Number Contracts'})
avg_village_revenue = village_revenue.merge(num_village_contracts, on = 'Village')
avg_village_revenue['Average Revenue'] = avg_village_revenue['Revenue']/avg_village_revenue['Number Contracts']
avg_village_revenue['Rank'] = avg_village_revenue['Average Revenue'].rank(ascending = False)
display(avg_village_revenue.sort_values('Rank').head(10))

In [ ]:
rec_w_village.columns

Index(['id_x', 'date', 'revenue', 'type', 'note', 'payment_wallet_name',
       'destination_type', 'destination_contract_reference',
       'origin_payment_id', 'origin_payment_transaction_id',
       'payment_wallet_id', 'lead_id', 'add_on_id', 'contract_payment_id',
       'contract_pending_payment_id', 'user_id', 'last_updated',
       'contract_reference', 'client_id', 'village'],
      dtype='object')